# Step 2 — Leakage-safe benchmark

We hold out entire bacterial strains. Every interaction belonging to a strain stays in exactly one of train, validation, or test. This prevents the model from seeing the target strain during training.

In [ ]:
from pathlib import Path
import pandas as pd

from phage_cocktail.benchmark import evaluate_broadest_cocktails, evaluate_pairwise_baselines
from phage_cocktail.data import load_interactions
from phage_cocktail.split import split_by_bacterium

DATA_PATH = Path('../data/raw/genophi_ecoli_subset.csv')
df = load_interactions(DATA_PATH)
assigned = split_by_bacterium(df, seed=42)
assigned.groupby('split')['bacterium'].nunique()

In [ ]:
assigned.groupby('split')['infection'].agg(['count', 'mean'])

## Pairwise baselines

These baselines deliberately know nothing about the held-out bacterium:

- **coin_flip_0.5**: predicts 0.5 for every pair.
- **global_prevalence**: predicts the training-set positive rate for every pair.
- **phage_prevalence**: predicts each phage's smoothed infection rate measured only on training strains.

A host-aware model should beat `phage_prevalence`, not merely beat chance.

In [ ]:
train = assigned[assigned['split'] == 'train'].drop(columns='split')
validation = assigned[assigned['split'] == 'validation'].drop(columns='split')
test = assigned[assigned['split'] == 'test'].drop(columns='split')

pairwise = evaluate_pairwise_baselines(train, test)
pairwise.round(4)

## Cocktail baseline: choose the broadest training-set phages

For cocktail size k, select the k phages with the highest training-strain hit rate. We compare:

1. the naive independence estimate, and
2. the actual fraction of completely unseen test strains infected by at least one selected phage.

A large gap is evidence that the phages are not independent—for example because their host ranges overlap.

In [ ]:
cocktails = evaluate_broadest_cocktails(train, test, cocktail_sizes=range(1, 6))
cocktails.round(4)

## Interpretation

On the fixed seed-42 split, phage prevalence alone is already a substantial predictor. That means future sequence/genome models must demonstrate incremental value beyond phage promiscuity.

The independence formula can also badly overestimate cocktail coverage because broad phages may infect many of the same strains. This redundancy gap is directly relevant to the project's central question: selecting combinations that add complementary coverage rather than repeatedly covering the same susceptible hosts.